In [5]:
import pandas as pd
import numpy as np 
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score


# 1. LOAD DATASET

In [40]:
df = pd.read_csv("Customer-Churn.csv")

# basic cleaning

In [43]:
df=df.drop("customerID",axis=1)

In [44]:
df["TotalCharges"]=pd.to_numeric(df["TotalCharges"],errors="coerce")
df["TotalCharges"]=df["TotalCharges"].fillna(df["TotalCharges"].median())

# ENCODE

In [45]:
df["Churn"]=df["Churn"].map({"Yes":1,"No":0})


In [46]:
# split

In [49]:
X=df.drop("Churn",axis=1)
y=df["Churn"]

In [50]:
cat_cols=X.select_dtypes(include="object").columns
num_cols=X.select_dtypes(exclude="object").columns

In [51]:
# preprocssing

In [52]:
numeric_pipeline=Pipeline([
    ("imputer",SimpleImputer(strategy="median"))
])
categorical_pipeline=Pipeline([                
    ("imputer",SimpleImputer(strategy="most_frequent")),
    ("encoder",OneHotEncoder(handle_unknown="ignore"))
])
preprocessor=ColumnTransformer([
    ("num",numeric_pipeline,num_cols),
    ("cat",categorical_pipeline,cat_cols)
])

# Train Test split

In [53]:
X_train,X_test,y_train,y_test=train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42)

# apply preproccesing

In [54]:
X_train_prep=preprocessor.fit_transform(X_train)
X_test_prep=preprocessor.transform(X_test)



In [55]:
# features selection

In [56]:
selector_model=RandomForestClassifier(
    n_estimators=300,
    random_state=42
)
selector_model.fit(X_train_prep,y_train)
selector=SelectFromModel(
    selector_model,
    threshold="median",
    prefit="True"
)
X_train_sel=selector.transform(X_train_prep)
X_test_sel=selector.transform(X_test_prep)
print("Original features:",X_train_prep.shape[1])
print("Selected features:",X_train_sel.shape[1])


Original features: 45
Selected features: 23


In [57]:
# final model

In [58]:
model=RandomForestClassifier(
    n_estimators=500,
    max_depth=10,
    class_weight="balanced",
    random_state=42
)
model.fit(X_train_sel,y_train)


,n_estimators,500
,criterion,'gini'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [59]:
# predict

In [60]:
pred=model.predict(X_test_sel)
prob=model.predict_proba(X_test_sel)[:,1]

In [62]:
pred,prob

(array([0, 1, 0, ..., 0, 0, 0], shape=(1409,), dtype=int64),
 array([0.01076946, 0.76620061, 0.10090156, ..., 0.23851246, 0.07100604,
        0.01529775], shape=(1409,)))